# 矩阵计算体验

# matmul_add 算子

## 概述

matmul_add 算子实现矩阵乘法与偏置加法的融合操作，计算公式为：

$$y(a, b, c) = a \times b^T + c$$

该算子基于 PyPTO 框架开发，支持 **m 轴动态**（`pypto.DYNAMIC`），即运行时可在不重新编译的前提下更换 m 维度大小。

## 算子规格

| 项目 | 说明 |
|------|------|
| 算子名称 | matmul_add |
| 计算公式 | `y = a @ b^T + c` |
| 数据类型 | bfloat16 |
| 精度标准 | atol=0.0001, rtol=0.0078125 |
| 动态轴 | m（第一维度） |

### 输入输出

| 参数 | 方向 | shape | dtype | 说明 |
|------|------|-------|-------|------|
| a | 输入 | [m, k] | bfloat16 | 左矩阵，m 为动态轴 |
| b | 输入 | [n, k] | bfloat16 | 右矩阵，计算时转置 |
| c | 输入 | [m, n] | bfloat16 | 偏置矩阵，m 为动态轴 |
| y | 输出 | [m, n] | bfloat16 | 计算结果 |

## 实现要点

### 核心代码结构

```python
@pypto.frontend.jit(runtime_options={"run_mode": global_run_mode})
def matmul_add_kernel(
    a: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),  # m 动态
    b: pypto.Tensor([], pypto.DT_BF16),                              # shape 不限定
    c: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),  # m 动态
    out: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),# m 动态
    n_val: int,                                                       # n 维度值
    k_val: int):                                                      # k 维度值
```

### 关键设计决策

#### 1. 动态轴声明

使用 `pypto.DYNAMIC` 标记 m 轴，使其可在不同调用间变化而无需重新编译。由于 DYNAMIC 张量不能直接作为算子操作数，必须在 `pypto.loop` 内通过 `pypto.view` 转为静态 shape 的 tile 后再计算。

#### 2. 仅对动态轴手动切分

只对动态的 m 轴进行手动 tiling（`pypto.loop` + `pypto.view` + `pypto.assemble`），静态的 n、k 轴由框架内部处理：

```python
for m_idx in pypto.loop(0, m_loop, 1, name="LOOP_m", idx_name="m_idx"):
    m_offset = m_idx * tile_m
    valid_m = (m_val - m_offset).min(tile_m)   # 符号表达式边界管理

    # view a, 为处理非对齐场景，需要手动设置validshape

    # view c, 为处理非对齐场景，需要手动设置validshape

    # mm_res = a @ b.T

    # add_result = mm_res + c_view

    pypto.assemble(add_result, [m_offset, 0], out)
```

**涉及的 PyPTO API**

| API | 用途 |
|-----|------|
| `pypto.frontend.jit` | Kernel JIT 编译装饰器 |
| `pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], ...)` | 声明动态/静态维度 |
| `pypto.loop` | 生成硬件级循环 |
| `pypto.view` | 从动态张量中切分静态 shape 的 tile |
| `pypto.assemble` | 将计算结果写回输出张量 |
| `pypto.set_cube_tile_shapes` | 设置矩阵乘法的 L0/L1 cache tile 尺寸 |
| `pypto.set_vec_tile_shapes` | 设置向量操作的 tile 尺寸 |
| `pypto.matmul(..., b_trans=True)` | 矩阵乘法，支持随路转置 |
| `pypto.add` | 元素级加法 |

参考文档：
- https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-view.md
- https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_cube_tile_shapes.md
- https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-matmul.md
- https://gitcode.com/cann/pypto/blob/master/docs/api/config/pypto-set_vec_tile_shapes.md
- https://gitcode.com/cann/pypto/blob/master/docs/api/operation/pypto-add.md

#### 3. b^T 通过 matmul 随路转置实现

使用 `pypto.matmul()` 中的 `b_trans=True` 参数，在矩阵乘法时随路完成 b 的转置，避免额外的数据搬移开销。

#### 4. Tile Shape 对齐

BF16 数据类型要求 `set_cube_tile_shapes` 的所有维度满足 **16 元素对齐**（即 32 字节 / 2 字节每元素）。当实际 k 或 n 不满足对齐时，向上取整：

```python
tile_k = ((k_val + 15) // 16) * 16
tile_n = ((n_val + 15) // 16) * 16
```

#### 5. 边界管理使用 `.min()` 方法

对于动态维度产生的符号表达式，**必须使用 `.min()` 方法**而非 Python 内置 `min()` 函数：

```python
# 正确：符号表达式的 .min() 方法
valid_m = (m_val - m_offset).min(tile_m)

# 错误：Python min() 无法处理符号表达式
# valid_m = min(m_offset + tile_m, m_val)
```

### DYNAMIC Loop 注意事项

PyPTO 的 DYNAMIC loop 在首次调用时确定循环次数并编译，后续调用：

- **可以**使用更少的迭代次数（通过 `valid_shape` 边界管理跳过多余迭代）
- **不可以**使用更多的迭代次数（超出编译时的循环次数）

因此，**首次调用应使用最大的 m 值**，以确保编译的循环次数覆盖后续所有场景。

## 测试用例

| 测试用例 | 说明 | 覆盖场景 |
|----------|------|----------|
| `test_matmul_add_basic` | 4x4 矩阵基础验证（b 为单位矩阵） | 基本功能正确性 |
| `test_matmul_add_dynamic_m` | m=128/64/32/16 动态轴验证（从大到小） | 动态 m 轴、多轮调用 |
| `test_matmul_add_non_square` | m=7, n=5, k=11 非对齐非方阵 | 非 16 对齐维度 |
| `test_matmul_add_edge_cases` | 全零输入 / 单行(m=1) / 大数值 | 边界值场景 |

In [1]:
#!/usr/bin/env python3
# coding: utf-8
# Copyright (c) 2025 Huawei Technologies Co., Ltd.
"""
matmul_add Operator: y = a @ b^T + c

This file implements the matmul_add PyPTO operator:
  y = matmul(a, b^T) + c

Inputs:
  - a: bfloat16 tensor, shape [m, k], m is dynamic
  - b: bfloat16 tensor, shape [n, k]
  - c: bfloat16 tensor, shape [m, n]
Output:
  - y: bfloat16 tensor, shape [m, n]

Precision: atol=0.0001, rtol=0.0078125
Dynamic axis: m (pypto.DYNAMIC)

Note on DYNAMIC loops:
    The kernel uses pypto.DYNAMIC for the m-axis. The loop iteration count is
    determined at the first kernel call. To support different m values, call
    with the largest m first so the compiled loop covers all subsequent calls.

Usage:
    python matmul_add.py                        # Run all tests
    python matmul_add.py --list                 # List available tests
    python matmul_add.py matmul_add::test_matmul_add_basic  # Run a specific test
"""

import argparse
import os
import sys
import pypto
import torch
import numpy as np
from numpy.testing import assert_allclose

import sys; sys.argv=['']
os.environ['TILE_FWK_DEVICE_ID'] = '0'


# ============================================================================
# matmul_add Kernel Definition
# ============================================================================

@pypto.frontend.jit(
    runtime_options={"run_mode": pypto.RunMode.NPU, "stitch_function_max_num": 128},
    debug_options={"runtime_debug_mode": 0, "compile_debug_mode": 0}
)
def matmul_add_kernel(
    a: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    b: pypto.Tensor([], pypto.DT_BF16),
    c: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    out: pypto.Tensor([pypto.DYNAMIC, pypto.STATIC], pypto.DT_BF16),
    n_val: int,
    k_val: int):
    """matmul_add kernel: computes y = a @ b^T + c.

    a is [m, k] where m is dynamic.
    b is [n, k] -> b^T is [k, n], so we use b_trans=True.
    c is [m, n].
    out is [m, n].

    m is declared as pypto.DYNAMIC, so the kernel can handle different m values
    without recompilation. Dynamic tensors must be accessed via pypto.view inside
    pypto.loop to get static-shape tiles for computation.

    Only the dynamic m-axis is manually tiled via loop; n and k are static and
    handled by the framework internally.

    Args:
        a: Input tensor A of shape [m, k], dtype bfloat16. m is dynamic.
        b: Input tensor B of shape [n, k], dtype bfloat16.
        c: Input tensor C of shape [m, n], dtype bfloat16. m is dynamic.
        out: Output tensor of shape [m, n], dtype bfloat16. m is dynamic.
        n_val: Static dimension n.
        k_val: Static dimension k.
    """
    m_val = a.shape[0]  # dynamic dimension -> symbolic scalar

    tile_m = 16
    # Cube tile shapes must satisfy BF16 16-element alignment (32 bytes)
    tile_k = ((k_val + 15) // 16) * 16
    tile_n = ((n_val + 15) // 16) * 16

    m_loop = (m_val + tile_m - 1) // tile_m

    for m_idx in pypto.loop(0, m_loop, 1, name="LOOP_m", idx_name="m_idx"):
        m_offset = m_idx * tile_m
        # Boundary: use .min() on symbolic expression (NOT Python's min())
        valid_m = (m_val - m_offset).min(tile_m)

        # View tiles along the dynamic m-axis
        a_view = pypto.view(a, [tile_m, k_val], [m_offset, 0],
                            valid_shape=[valid_m, k_val])
        c_view = pypto.view(c, [tile_m, n_val], [m_offset, 0],
                            valid_shape=[valid_m, n_val])

        # Set cube tile shapes for matmul
        pypto.set_cube_tile_shapes([128, 128], [128, 128], [128, 128])

        # matmul: a_view [tile_m, k_val] @ b^T [k_val, n_val] -> [tile_m, n_val]
        mm_result = pypto.matmul(a_view, b, pypto.DT_BF16, b_trans=True)

        # add bias: mm_result + c_view
        pypto.set_vec_tile_shapes(tile_m, tile_n)
        # TODO: 实现 add_result = mm_result + c_view
        add_result = pypto.add(mm_result, c_view)

        # Assemble result back to output
        pypto.assemble(add_result, [m_offset, 0], out)


# ============================================================================
# Test Cases
# ============================================================================

def test_matmul_add_basic(device_id: int = None):
    """Test basic usage of matmul_add operator: y = a @ b^T + c."""
    print("=" * 60)
    print("Test: Basic Usage of matmul_add Operator")
    print("=" * 60)

    device = f'npu:{device_id}'

    dtype = torch.bfloat16
    m, n, k = 4, 4, 4

    a = torch.tensor([[1.0, 2.0, 3.0, 4.0],
                      [5.0, 6.0, 7.0, 8.0],
                      [9.0, 10.0, 11.0, 12.0],
                      [13.0, 14.0, 15.0, 16.0]], dtype=dtype, device=device)
    b = torch.tensor([[1.0, 0.0, 0.0, 0.0],
                      [0.0, 1.0, 0.0, 0.0],
                      [0.0, 0.0, 1.0, 0.0],
                      [0.0, 0.0, 0.0, 1.0]], dtype=dtype, device=device)
    c = torch.tensor([[0.1, 0.2, 0.3, 0.4],
                      [0.5, 0.6, 0.7, 0.8],
                      [0.9, 1.0, 1.1, 1.2],
                      [1.3, 1.4, 1.5, 1.6]], dtype=dtype, device=device)

    # Golden: y = a @ b^T + c = a @ I + c = a + c (since b is identity)
    expected = a @ b.T + c

    out = torch.empty((m, n), dtype=dtype, device=device)
    matmul_add_kernel(a, b, c, out, n, k)
    assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                    rtol=0.0078125, atol=0.0001)
    print(f"Input a:      {a}")
    print(f"Input b:      {b}")
    print(f"Input c:      {c}")
    print(f"Output:       {out}")
    print(f"Expected:     {expected}")
    print("✓ Basic usage of matmul_add operator completed successfully")


def test_matmul_add_dynamic_m(device_id: int = None):
    """Test matmul_add operator with dynamic m-axis: different m sizes at runtime."""
    print("=" * 60)
    print("Test: matmul_add Operator - Dynamic m-axis")
    print("=" * 60)

    device = f'npu:{device_id}'

    dtype = torch.bfloat16
    n, k = 32, 64

    # Test multiple values of m to verify dynamic axis support
    # IMPORTANT: iterate from largest to smallest m. PyPTO DYNAMIC loops compile
    # for the iteration count of the first call; subsequent calls with fewer
    # iterations work via valid_shape boundary management.
    for m in [128, 64, 32, 16]:
        a = torch.randn(m, k, dtype=dtype, device=device)
        b = torch.randn(n, k, dtype=dtype, device=device)
        c = torch.randn(m, n, dtype=dtype, device=device)

        expected = a @ b.T + c

        out = torch.empty((m, n), dtype=dtype, device=device)
        matmul_add_kernel(a, b, c, out, n, k)
        assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                        rtol=0.0078125, atol=0.0001)

        max_diff = np.abs(out.cpu().float().numpy() - expected.cpu().float().numpy()).max()
        print(f"  m={m:4d}, n={n:2d}, k={k:2d}: max_diff={max_diff:.8f}  ✓")

    print("✓ Dynamic m-axis test completed successfully")


def test_matmul_add_non_square(device_id: int = None):
    """Test matmul_add with non-square matrices and non-16-aligned dimensions."""
    print("=" * 60)
    print("Test: matmul_add Operator - Non-square Matrices")
    print("=" * 60)

    device = f'npu:{device_id}'

    dtype = torch.bfloat16

    # Non-square: m=7, n=5, k=11 (not aligned to tile sizes)
    m, n, k = 7, 5, 11

    a = torch.randn(m, k, dtype=dtype, device=device)
    b = torch.randn(n, k, dtype=dtype, device=device)
    c = torch.randn(m, n, dtype=dtype, device=device)

    expected = a @ b.T + c

    out = torch.empty((m, n), dtype=dtype, device=device)
    matmul_add_kernel(a, b, c, out, n, k)
    assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                    rtol=0.0078125, atol=0.0001)

    max_diff = np.abs(out.cpu().float().numpy() - expected.cpu().float().numpy()).max()
    print(f"  m={m}, n={n}, k={k}: max_diff={max_diff:.8f}  ✓")
    print("✓ Non-square matrices test completed successfully")


def test_matmul_add_edge_cases(device_id: int = None):
    """Test matmul_add with edge cases: zeros, single row, large values."""
    print("=" * 60)
    print("Test: matmul_add Operator - Edge Cases")
    print("=" * 60)

    device = f'npu:{device_id}'
    dtype = torch.bfloat16

    # 1. All zeros - output should equal c
    m, n, k = 4, 4, 4
    a = torch.zeros((m, k), dtype=dtype, device=device)
    b = torch.zeros((n, k), dtype=dtype, device=device)
    c = torch.randn(m, n, dtype=dtype, device=device)
    expected = a @ b.T + c  # should equal c
    out = torch.empty((m, n), dtype=dtype, device=device)
    matmul_add_kernel(a, b, c, out, n, k)
    assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                    rtol=0.0078125, atol=0.0001)
    print("  [zeros]    a=zeros, b=zeros, c=random    ✓")

    # 2. Single row (m=1) - minimum dynamic size
    m, n, k = 1, 8, 16
    a = torch.randn(m, k, dtype=dtype, device=device)
    b = torch.randn(n, k, dtype=dtype, device=device)
    c = torch.randn(m, n, dtype=dtype, device=device)
    expected = a @ b.T + c
    out = torch.empty((m, n), dtype=dtype, device=device)
    matmul_add_kernel(a, b, c, out, n, k)
    assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                    rtol=0.0078125, atol=0.0001)
    print("  [m=1]      single row matrix              ✓")

    # 3. Large values
    m, n, k = 4, 4, 4
    a = torch.tensor([[100.0, -200.0, 300.0, -400.0],
                      [500.0, -600.0, 700.0, -800.0],
                      [-50.0, 60.0, -70.0, 80.0],
                      [10.0, -20.0, 30.0, -40.0]], dtype=dtype, device=device)
    b = torch.tensor([[10.0, 20.0, 30.0, 40.0],
                      [-10.0, -20.0, -30.0, -40.0],
                      [5.0, 10.0, 15.0, 20.0],
                      [-5.0, -10.0, -15.0, -20.0]], dtype=dtype, device=device)
    c = torch.randn(m, n, dtype=dtype, device=device)
    expected = a @ b.T + c
    out = torch.empty((m, n), dtype=dtype, device=device)
    matmul_add_kernel(a, b, c, out, n, k)
    assert_allclose(out.cpu().float().numpy(), expected.cpu().float().numpy(),
                    rtol=0.0078125, atol=0.0001)
    print("  [large]    large absolute values          ✓")

    print("✓ Edge cases test completed successfully")


# ============================================================================
# Main entry point
# ============================================================================

def main():
    """Run matmul_add operator tests.

    Usage:
        python matmul_add.py                        # Run all tests
        python matmul_add.py --list                 # List all available tests
        python matmul_add.py matmul_add::test_matmul_add_basic  # Run a specific case
    """
    parser = argparse.ArgumentParser(
        description="PyPTO matmul_add Operator Tests",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog="""
Examples:
  %(prog)s                                    Run all tests
  %(prog)s --list                             List all available tests
  %(prog)s matmul_add::test_matmul_add_basic    Run a specific test
        """
    )
    parser.add_argument(
        'example_id',
        type=str,
        nargs="?",
        help='Run a specific test case (e.g., matmul_add::test_matmul_add_basic). If omitted, run all tests.'
    )
    parser.add_argument(
        '--list',
        action='store_true',
        help='List all available tests and exit'
    )

    args = parser.parse_args()

    # Registry of all test cases
    examples = {
        'matmul_add::test_matmul_add_basic': {
            'name': 'Basic usage of matmul_add operator',
            'description': 'Verify y = a @ b^T + c with a fixed small tensor (identity b).',
            'function': test_matmul_add_basic
        },
        'matmul_add::test_matmul_add_dynamic_m': {
            'name': 'Dynamic m-axis test',
            'description': 'Verify the operator handles different m sizes at runtime.',
            'function': test_matmul_add_dynamic_m
        },
        'matmul_add::test_matmul_add_non_square': {
            'name': 'Non-square matrices test',
            'description': 'Verify correctness with non-square matrices and non-aligned dimensions.',
            'function': test_matmul_add_non_square
        },
        'matmul_add::test_matmul_add_edge_cases': {
            'name': 'Edge cases (zeros, single row, large values)',
            'description': 'Verify correctness on boundary inputs.',
            'function': test_matmul_add_edge_cases
        },
    }

    if args.list:
        print("\n" + "=" * 60)
        print("Available Tests for matmul_add Operator")
        print("=" * 60 + "\n")
        for case_key, ex_info in sorted(examples.items()):
            print(f"  {case_key}")
            print(f"     Name: {ex_info['name']}")
            print(f"     Description: {ex_info['description']}\n")
        return

    # Select tests to run
    if args.example_id:
        if args.example_id not in examples:
            print(f"ERROR: Invalid case '{args.example_id}'")
            print(f"Valid cases are: {', '.join(sorted(examples.keys()))}")
            print("\nUse --list to see all available tests.")
            sys.exit(1)
        examples_to_run = [(args.example_id, examples[args.example_id])]
    else:
        examples_to_run = list(examples.items())

    print("\n" + "=" * 60)
    print("PyPTO matmul_add Operator Tests")
    print("=" * 60 + "\n")

    import torch_npu
    torch.npu.set_device(0)
    device_id = 0

    try:
        for case_key, ex_info in examples_to_run:
            ex_info['function'](device_id)

        if len(examples_to_run) > 1:
            print("=" * 60)
            print("All matmul_add tests passed!")
            print("=" * 60)

    except Exception as e:
        print(f"\nError: {e}")
        raise


if __name__ == "__main__":
    main()


PyPTO matmul_add Operator Tests

Test: Basic Usage of matmul_add Operator


[W521 10:32:26.593185237 compiler_depend.ts:164] Warning: Device do not support double dtype now, dtype cast replace with float. (function operator())


Input a:      tensor([[ 1.,  2.,  3.,  4.],
        [ 5.,  6.,  7.,  8.],
        [ 9., 10., 11., 12.],
        [13., 14., 15., 16.]], device='npu:0', dtype=torch.bfloat16)
Input b:      tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]], device='npu:0', dtype=torch.bfloat16)
Input c:      tensor([[0.1001, 0.2002, 0.3008, 0.4004],
        [0.5000, 0.6016, 0.6992, 0.8008],
        [0.8984, 1.0000, 1.1016, 1.2031],
        [1.2969, 1.3984, 1.5000, 1.6016]], device='npu:0',
       dtype=torch.bfloat16)
Output:       tensor([[ 1.1016,  2.2031,  3.2969,  4.4062],
        [ 5.5000,  6.5938,  7.6875,  8.8125],
        [ 9.8750, 11.0000, 12.1250, 13.1875],
        [14.3125, 15.3750, 16.5000, 17.6250]], device='npu:0',
       dtype=torch.bfloat16)
Expected:     tensor([[ 1.1016,  2.2031,  3.2969,  4.4062],
        [ 5.5000,  6.5938,  7.6875,  8.8125],
        [ 9.8750, 11.0000, 12.1250, 13.1875],
        [14.3125, 15.3750, 16.5000, 17.6250]],

# 用户需求(For Ai AGent)

本文档记录了用户开发算子的需求，请根据描述进行需求检测及分析。

---

## 需求-1
请开发一个 PyPTO 算子：
- 算子名称：matmul_add
- 公式：$ y(a, b, c) = a @ b^T + c $
- 规格：

| 类型  | shape  | dtype  |
| ------------ | ------------ | ------------ |
| 输入 a| \[m, k\] | bfloat16  |
| 输入 b| \[n, k\] | bfloat16  |
| 输入 c| \[m, n\] | bfloat16  |
| 输出 y| \[m, n\] | bfloat16  |

- 精度标准：atol=0.0001, rtol=0.0078125
- 其中要求，kernel本身是动态的，m是动态轴

---